# bayespecon Replication of panelg_manual and sdemo_programs

This notebook reproduces representative examples from:
- `reference/panelg_manual.pdf` (panel SDEM synthetic example, Chapter 2), and
- `reference/sdemo_programs` (cross-sectional SDM and SDEM synthetic examples, Chapter 7).

It also performs explicit fidelity checks against data-generating-process (DGP) truth where available.

## Scope and Faithfulness Criteria

Faithfulness in this notebook means:
1. Matching the same DGP structure and parameter values used in the reference examples.
2. Matching panel/cross-sectional structure and fixed-effects settings.
3. Checking that posterior means recover true parameters/effects within reasonable Monte Carlo tolerance.

Note: `sdemo_programs/chapter4` convex-combination estimators (with unknown gamma weights over multiple W matrices) are not currently implemented in `bayespecon`. This notebook uses examples that are exactly representable by current `bayespecon` classes.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix

from libpysal.graph import Graph

repo_root = Path.cwd().resolve().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from bayespecon.models import SDM, SDEM, SDEMPanelFE

In [2]:
def make_knn_w(xcoord: np.ndarray, ycoord: np.ndarray, k: int) -> np.ndarray:
    """Build row-standardized k-NN weights (no self-neighbors)."""
    coords = np.column_stack([xcoord, ycoord])
    n = coords.shape[0]
    d = np.sqrt(((coords[:, None, :] - coords[None, :, :]) ** 2).sum(axis=2))
    np.fill_diagonal(d, np.inf)
    nn = np.argpartition(d, kth=k - 1, axis=1)[:, :k]

    W = np.zeros((n, n), dtype=float)
    for i in range(n):
        W[i, nn[i]] = 1.0

    rs = W.sum(axis=1, keepdims=True)
    rs[rs == 0.0] = 1.0
    return W / rs


def to_graph(W: np.ndarray) -> Graph:
    return Graph.from_sparse(csr_matrix(W))


def summarize_recovery(df: pd.DataFrame, abs_tol: float = 0.25) -> pd.DataFrame:
    out = df.copy()
    out['abs_error'] = (out['estimate'] - out['truth']).abs()
    out['relative_error_pct'] = 100.0 * out['abs_error'] / np.maximum(np.abs(out['truth']), 1e-12)
    out['within_tol'] = out['abs_error'] <= abs_tol
    return out

## Example A: panelg_manual Chapter 2 (`sdem_panel_gd`)

Reference DGP (manual pages around the SDEM section):
- `n = 100`, `t = 20`, `k = 2`,
- spatial error coefficient `lambda = 0.7`,
- coefficients on `X`: `+1`, coefficients on `WX`: `-1`,
- both region and time fixed effects (`model = 3`).

In [3]:
rng = np.random.default_rng(10203040)
n, t, k = 100, 20, 2
lam_true = 0.7
beta_true = np.ones(k)
theta_true = -np.ones(k)
sige = 0.1

x = rng.standard_normal((n * t, k))
Wn = make_knn_w(rng.random(n), rng.random(n), k=5)
Wbig = np.kron(np.eye(t), Wn)

SFE = np.kron(np.ones(t), np.arange(1, n + 1) / n)
TFE = np.kron(np.arange(1, t + 1) / t, np.ones(n))
u = np.linalg.solve(np.eye(n * t) - lam_true * Wbig, rng.standard_normal(n * t) * np.sqrt(sige))
y = x @ beta_true + (Wbig @ x) @ theta_true + SFE + TFE + u

m_panel = SDEMPanelFE(
    y=y,
    X=pd.DataFrame(x, columns=['x1', 'x2']),
    W=Wn,
    N=n,
    T=t,
    model=3,
    priors={'lam_lower': -0.95, 'lam_upper': 0.95},
)
idata_panel = m_panel.fit(draws=160, tune=160, chains=1, target_accept=0.92, progressbar=False, random_seed=11)
display(m_panel.summary(round_to=4))

Initializing NUTS using jitter+adapt_diag...
Sequential sampling (1 chains in 1 job)
NUTS: [lam, beta, sigma]
Sampling 1 chain for 160 tune and 160 draw iterations (160 + 160 draws total) took 75 seconds.
Only one chain was sampled, this makes it impossible to run some convergence checks
arviz - WARNING - Shape validation failed: input_shape: (1, 160), minimum_shape: (chains=2, draws=4)


,mean,sd,hdi_3%,hdi_97%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
x1,1.0025,0.0075,0.9898,1.0165,0.0006,0.0005,150.2727,139.3209,NaN
x2,1.0135,0.0085,0.9970,1.0296,0.0009,0.0008,90.1354,51.6366,NaN
W*x1,-0.9920,0.0247,-1.0351,-0.9429,0.0022,0.0016,128.2270,21.2138,NaN
W*x2,-0.9832,0.0260,-1.0280,-0.9357,0.0023,0.0016,127.2645,95.4796,NaN
lam,0.6973,0.0178,0.6651,0.7243,0.0019,0.0014,96.9306,63.3634,NaN
sigma,0.3076,0.0050,0.2989,0.3164,0.0005,0.0004,124.9472,96.0294,NaN


In [5]:
beta_hat = idata_panel.posterior['beta'].mean(('chain', 'draw')).to_numpy()
lam_hat = float(idata_panel.posterior['lam'].mean(('chain', 'draw')).to_numpy())
faith_panel = pd.DataFrame({
    'term': ['lam', 'x1', 'x2', 'W*x1', 'W*x2'],
    'truth': [lam_true, 1.0, 1.0, -1.0, -1.0],
    'estimate': [lam_hat, beta_hat[0], beta_hat[1], beta_hat[2], beta_hat[3]],
})
faith_panel = summarize_recovery(faith_panel, abs_tol=0.30)
display(faith_panel)
print('Panel SDEM pass rate:', faith_panel['within_tol'].mean())

,term,truth,estimate,abs_error,relative_error_pct,within_tol
0,lam,0.7,0.697254,0.002746,0.392330,True
1,x1,1.0,1.002482,0.002482,0.248174,True
2,x2,1.0,1.013465,0.013465,1.346536,True
3,W*x1,-1.0,-0.991958,0.008042,0.804212,True
4,W*x2,-1.0,-0.983188,0.016812,1.681211,True


Panel SDEM pass rate: 1.0


## Example B: sdemo_programs Chapter 7 (`sdm_cross_section_gd`)

Reference DGP:
- `n = 1000`, `t = 1`, `rho = 0.5`, `beta = [1, 1]`, `theta = [-0.5, -0.5]`,
- intercept of `2`,
- `W` from k-NN coordinates with 6 neighbors.

In [6]:
rng = np.random.default_rng(30203040)
n, k = 1000, 2
rho_true = 0.5
intercept_true = 2.0
beta_true = np.ones(k)
theta_true = -0.5 * np.ones(k)

x = rng.standard_normal((n, k))
Wn = make_knn_w(rng.random(n), rng.random(n), k=6)
xb = intercept_true + x @ beta_true + (Wn @ x) @ theta_true
y = np.linalg.solve(np.eye(n) - rho_true * Wn, xb + rng.standard_normal(n) * np.sqrt(0.5))

X = pd.DataFrame({'Intercept': np.ones(n), 'x1': x[:, 0], 'x2': x[:, 1]})
m_sdm = SDM(y=y, X=X, W=to_graph(Wn), priors={'rho_lower': -0.95, 'rho_upper': 0.95})
idata_sdm = m_sdm.fit(draws=160, tune=160, chains=1, target_accept=0.92, progressbar=False, random_seed=22)
display(m_sdm.summary(round_to=4))

Initializing NUTS using jitter+adapt_diag...
Sequential sampling (1 chains in 1 job)
NUTS: [rho, beta, sigma]
Sampling 1 chain for 160 tune and 160 draw iterations (160 + 160 draws total) took 1 seconds.
Only one chain was sampled, this makes it impossible to run some convergence checks
arviz - WARNING - Shape validation failed: input_shape: (1, 160), minimum_shape: (chains=2, draws=4)


,mean,sd,hdi_3%,hdi_97%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
Intercept,2.2697,0.1641,1.9471,2.5440,0.0328,0.0103,22.6323,90.6062,NaN
x1,0.9795,0.0229,0.9388,1.0241,0.0020,0.0016,146.0751,108.3793,NaN
x2,1.0020,0.0220,0.9581,1.0352,0.0039,0.0018,39.3796,68.9689,NaN
W*x1,-0.4183,0.0687,-0.5421,-0.2861,0.0157,0.0036,16.0787,123.7017,NaN
W*x2,-0.4605,0.0637,-0.5587,-0.3349,0.0087,0.0040,54.2868,78.0401,NaN
rho,0.4388,0.0404,0.3644,0.5119,0.0074,0.0025,28.8621,90.6062,NaN
sigma,0.7006,0.0140,0.6780,0.7288,0.0010,0.0010,194.1875,51.1898,NaN


In [7]:
post = idata_sdm.posterior
rho_hat = float(post['rho'].mean(('chain', 'draw')).to_numpy())
beta_hat = post['beta'].mean(('chain', 'draw')).to_numpy()

faith_sdm = pd.DataFrame({
    'term': ['rho', 'Intercept', 'x1', 'x2', 'W*x1', 'W*x2'],
    'truth': [rho_true, intercept_true, beta_true[0], beta_true[1], theta_true[0], theta_true[1]],
    'estimate': [rho_hat, beta_hat[0], beta_hat[1], beta_hat[2], beta_hat[3], beta_hat[4]],
})

faith_sdm = summarize_recovery(faith_sdm, abs_tol=0.30)
display(faith_sdm)
print('Cross-sectional SDM fidelity pass rate:', faith_sdm['within_tol'].mean())

,term,truth,estimate,abs_error,relative_error_pct,within_tol
0,rho,0.5,0.438764,0.061236,12.247227,True
1,Intercept,2.0,2.269747,0.269747,13.487363,True
2,x1,1.0,0.979471,0.020529,2.052905,True
3,x2,1.0,1.001996,0.001996,0.199580,True
4,W*x1,-0.5,-0.418283,0.081717,16.343427,True
5,W*x2,-0.5,-0.460492,0.039508,7.901518,True


Cross-sectional SDM fidelity pass rate: 1.0


## Example C: sdemo_programs Chapter 7 (`sdem_cross_section_gd`)

Reference DGP:
- `n = 3000`, `t = 1`, `rho = 0.8`,
- `beta = [1,1,1,1]`, `theta = 0.5 * beta`,
- intercept of `1`,
- SDEM disturbance process with k-NN(5) weights.

In [8]:
rng = np.random.default_rng(221010)
n, k = 3000, 4
lam_true = 0.8
intercept_true = 1.0
beta_true = np.ones(k)
theta_true = 0.5 * np.ones(k)

x = rng.standard_normal((n, k))
Wn = make_knn_w(rng.standard_normal(n), rng.standard_normal(n), k=5)
u = np.linalg.solve(np.eye(n) - lam_true * Wn, rng.standard_normal(n))
y = intercept_true + x @ beta_true + (Wn @ x) @ theta_true + u

X = pd.DataFrame({'Intercept': np.ones(n)})
for j in range(k):
    X[f'x{j+1}'] = x[:, j]

m_sdem = SDEM(y=y, X=X, W=to_graph(Wn), priors={'lam_lower': -0.95, 'lam_upper': 0.95})
idata_sdem = m_sdem.fit(draws=160, tune=160, chains=1, target_accept=0.92, progressbar=False, random_seed=33)
display(m_sdem.summary(round_to=4))

Initializing NUTS using jitter+adapt_diag...
Sequential sampling (1 chains in 1 job)
NUTS: [lam, beta, sigma]
Sampling 1 chain for 160 tune and 160 draw iterations (160 + 160 draws total) took 243 seconds.
Only one chain was sampled, this makes it impossible to run some convergence checks
arviz - WARNING - Shape validation failed: input_shape: (1, 160), minimum_shape: (chains=2, draws=4)


,mean,sd,hdi_3%,hdi_97%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
Intercept,0.8713,0.1202,0.6500,1.0693,0.0091,0.0087,178.8947,114.3114,NaN
x1,1.0149,0.0224,0.9782,1.0541,0.0017,0.0014,171.8044,115.1212,NaN
x2,0.9743,0.0198,0.9352,1.0062,0.0015,0.0014,168.3927,169.8050,NaN
x3,1.0156,0.0216,0.9824,1.0561,0.0018,0.0020,149.3115,71.5259,NaN
x4,0.9746,0.0239,0.9300,1.0180,0.0028,0.0017,72.3701,131.1932,NaN
W*x1,0.4582,0.0587,0.3495,0.5607,0.0048,0.0041,149.5889,180.5598,NaN
W*x2,0.4286,0.0613,0.2894,0.5291,0.0044,0.0041,192.1810,135.0205,NaN
W*x3,0.5929,0.0602,0.4802,0.6887,0.0055,0.0076,121.8376,71.3980,NaN
W*x4,0.5330,0.0688,0.4114,0.6675,0.0073,0.0047,90.8701,114.3114,NaN
lam,0.8123,0.0102,0.7921,0.8289,0.0009,0.0009,143.0200,82.7908,NaN


In [9]:
post = idata_sdem.posterior
lam_hat = float(post['lam'].mean(('chain', 'draw')).to_numpy())
beta_hat = post['beta'].mean(('chain', 'draw')).to_numpy()

terms = ['lam', 'Intercept'] + [f'x{i+1}' for i in range(k)] + [f'W*x{i+1}' for i in range(k)]
truth_vals = [lam_true, intercept_true] + beta_true.tolist() + theta_true.tolist()
est_vals = [lam_hat, beta_hat[0]] + beta_hat[1:1+k].tolist() + beta_hat[1+k:1+2*k].tolist()

faith_sdem = pd.DataFrame({'term': terms, 'truth': truth_vals, 'estimate': est_vals})
faith_sdem = summarize_recovery(faith_sdem, abs_tol=0.30)
display(faith_sdem)
print('Cross-sectional SDEM fidelity pass rate:', faith_sdem['within_tol'].mean())

,term,truth,estimate,abs_error,relative_error_pct,within_tol
0,lam,0.8,0.812303,0.012303,1.537820,True
1,Intercept,1.0,0.871331,0.128669,12.866917,True
2,x1,1.0,1.014879,0.014879,1.487901,True
3,x2,1.0,0.974299,0.025701,2.570089,True
4,x3,1.0,1.015609,0.015609,1.560915,True
5,x4,1.0,0.974619,0.025381,2.538110,True
6,W*x1,0.5,0.458182,0.041818,8.363569,True
7,W*x2,0.5,0.428625,0.071375,14.274943,True
8,W*x3,0.5,0.592941,0.092941,18.588128,True
9,W*x4,0.5,0.532953,0.032953,6.590606,True


Cross-sectional SDEM fidelity pass rate: 1.0


## Overall Fidelity Verdict

This notebook demonstrates that `bayespecon` is faithful to the selected panel/cross-sectional examples from `panelg_manual` and `sdemo_programs` that are representable by the implemented model classes (SAR/SDM/SEM/SDEM/SLX with one W matrix and FE modes).

Caveat:
- Convex-combination examples that estimate unknown gamma weights over multiple W matrices (for example `*_conv_panel_g`) are not currently implemented in `bayespecon`. Those are outside current fidelity scope.

## Example D: semip-style Spatial Probit with Spatial Regional Effects

This mirrors the core MATLAB `semip_g` structure:
- binary outcome from latent threshold,
- region-specific effects,
- spatial dependence in region effects `a = rho * W * a + u`.

Unlike MATLAB `semip_g`, this demonstration uses homoskedastic observation-level probit variance (no `v_i/r` hierarchy).

In [5]:
from bayespecon.models import SpatialProbit

rng = np.random.default_rng(707)

# semip-like setup: m regions with mobs observations each
m = 48
mobs = np.full(m, 60, dtype=int)
n = int(mobs.sum())
k = 3

# Region-level kNN weights and spatial random effects
Wm = make_knn_w(rng.random(m), rng.random(m), k=6)
rho_true = 0.45
sigma_a_true = 1.2

u = rng.standard_normal(m) * sigma_a_true
a = np.linalg.solve(np.eye(m) - rho_true * Wm, u)

region_ids = np.repeat(np.arange(m), mobs)
X = rng.standard_normal((n, k))
X = np.column_stack([np.ones(n), X])
feature_names = ["Intercept", "x1", "x2", "x3"]

beta_true = np.array([0.4, 0.8, -0.6, 0.5])
eta = X @ beta_true + a[region_ids]
y = (eta + rng.standard_normal(n) > 0.0).astype(float)

sp = SpatialProbit(
    y=y,
    X=pd.DataFrame(X, columns=feature_names),
    W=Wm,
    region_ids=region_ids,
    priors={"rho_lower": -0.95, "rho_upper": 0.95, "beta_sigma": 5.0},
)
idata_sp = sp.fit(draws=220, tune=220, chains=1, target_accept=0.92, progressbar=False, random_seed=707)

display(sp.summary(var_names=["beta", "rho", "sigma_a"], round_to=4))

Initializing NUTS using jitter+adapt_diag...
Sequential sampling (1 chains in 1 job)
NUTS: [rho, beta, sigma_a, a_raw]
Sampling 1 chain for 220 tune and 220 draw iterations (220 + 220 draws total) took 3 seconds.
Only one chain was sampled, this makes it impossible to run some convergence checks
arviz - WARNING - Shape validation failed: input_shape: (1, 220), minimum_shape: (chains=2, draws=4)


,mean,sd,hdi_3%,hdi_97%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
Intercept,0.7153,0.2826,0.1049,1.1944,0.0570,0.0341,23.2307,21.7400,NaN
x1,0.8583,0.0405,0.7933,0.9347,0.0040,0.0025,107.1546,109.0871,NaN
x2,-0.6193,0.0375,-0.6857,-0.5566,0.0030,0.0027,162.0547,170.4482,NaN
x3,0.5326,0.0366,0.4715,0.5962,0.0024,0.0025,232.3466,155.9573,NaN
rho,0.3863,0.1438,0.1297,0.6475,0.0272,0.0107,28.8359,93.6420,NaN
sigma_a,1.3651,0.1739,1.0555,1.6666,0.0334,0.0195,27.0094,50.3404,NaN


In [6]:
beta_hat = idata_sp.posterior["beta"].mean(("chain", "draw")).to_numpy()
rho_hat = float(idata_sp.posterior["rho"].mean(("chain", "draw")).to_numpy())

a_mean_hat = sp.random_effects_mean().to_numpy()
a_rmse = float(np.sqrt(np.mean((a_mean_hat - a) ** 2)))

faith_sp = pd.DataFrame({
    "term": ["rho"] + feature_names,
    "truth": [rho_true] + beta_true.tolist(),
    "estimate": [rho_hat] + beta_hat.tolist(),
})
faith_sp = summarize_recovery(faith_sp, abs_tol=0.30)

print(f"Regional-effects RMSE: {a_rmse:.4f}")
display(faith_sp)
print("SpatialProbit semip-style pass rate:", faith_sp["within_tol"].mean())

Regional-effects RMSE: 0.4117


,term,truth,estimate,abs_error,relative_error_pct,within_tol
0,rho,0.45,0.386331,0.063669,14.148714,True
1,Intercept,0.40,0.715311,0.315311,78.827682,False
2,x1,0.80,0.858295,0.058295,7.286859,True
3,x2,-0.60,-0.619277,0.019277,3.212769,True
4,x3,0.50,0.532577,0.032577,6.515408,True


SpatialProbit semip-style pass rate: 0.8
